# PIP agreement after merging ADSP and UK Biobank study sketches

Inputs: the ADSP + UKB meta-analysis simulations of `12` `[meta_mix_simulation]` (`ADSP_UKB_Mix_simulation`, total n = 2,000). The study genotypes are used when the `12` output stores them (`UKB_Xmat`, `ADSP_Xmat`); otherwise the two cohorts are re-simulated from the stored full panels with the same causal variants, as `12` does.

For each replicate, baseline SuSiE-RSS is fitted to the sample-size weighted meta-analysis z with

- the in-sample meta-analysis LD, R_meta = Σ n_k R_k / N (Eq. S54), and
- the merged raw Gaussian study sketches, U_meta = [√(n₁/B) U₁, √(n₂/B) U₂] (Eq. 14 / S55), at each study sketch dimension B.

Save both PIP vectors and credible sets for each replicate and B, along with PIP correlation, maximum absolute PIP difference, and LD MSE for the merged sketch.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Merged_study_sketch_PIP/result/h2g_<h2g>/causal_<k>/prop_<prop>/`

In [ ]:
[merged_sketch_pip]
# Fig S8: baseline SuSiE-RSS on the ADSP + UKB meta-analysis z (12 [meta_mix_simulation]) with
#   (a) in-sample meta-analysis LD   R_meta = sum_k n_k R_k / N                                   (Eq. S54)
#   (b) merged raw Gaussian study sketches U_meta = [sqrt(n_1/B) U_1, sqrt(n_2/B) U_2], R = U_meta U_meta' / N   (Eq. 14 / S55)
# for each study-sketch dimension B, and compare variant PIPs.
parameter: sim_files = paths
# 12 output files of one (h2g, causal, prop) setting; prop must give both studies (e.g. 0.50)
parameter: h2g = 0.04
# used only if the cohorts have to be re-simulated (study genotypes not stored in the 12 output)
parameter: B = [500, 1000, 2000, 5000]
# study-sketch dimension(s), the same for both studies
parameter: seed = 2026
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "50h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: sim_files, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_merged_sketch.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(Rfast)
    library(susieR)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    select <- dplyr::select
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
    `%||%` <- function(a, b) if (is.null(a)) b else a

    path <- ${_input:r}
    dats <- readRDS(path)
    idx  <- as.integer(str_match(basename(path), "sample_([0-9]+)_")[, 2])
    set.seed(${seed} + idx)

    ## ---- the two cohorts: study genotypes + z ------------------------------------------
    zfrom <- function(X, y) { rr <- susieR::univariate_regression(X, y); rr$betahat / rr$sebetahat }
    ## exact names: dats$UKB_Xmat would partially match UKB_Xmat_full (the full panel)
    if (all(c("UKB_Xmat", "ADSP_Xmat") %in% names(dats))) {
      ## study genotypes stored by 12: use them with their sumstats
      X_U <- dats[["UKB_Xmat"]]; X_A <- dats[["ADSP_Xmat"]]
      ref_order <- colnames(X_U); X_A <- X_A[, ref_order, drop = FALSE]
      z_U <- dats$UKB_target_tb$z[match(ref_order, dats$UKB_target_tb$variant_id)]
      z_A <- dats$ADSP_target_tb$z[match(ref_order, dats$ADSP_target_tb$variant_id)]
      true_vars <- match(colnames(dats[["UKB_Xmat_full"]] %||% X_U)[dats$true_vars], ref_order)
      source_of_cohorts <- "12 (stored)"
    } else {
      ## re-simulate the two cohorts from the stored full panels, as 12 [meta_mix_simulation] does
      U_full <- dats[["UKB_Xmat_full"]]; A_full <- dats[["ADSP_Xmat_full"]]
      ref_order <- colnames(U_full); A_full <- A_full[, ref_order, drop = FALSE]
      true_vars <- dats$true_vars
      B_mat <- matrix(0, ncol(U_full), 1)
      B_mat[, 1] <- sim_beta_fix_variant(G = U_full, causal_index = true_vars, is_h2g_total = FALSE)
      draw <- function(G, n) {
        X <- G[sample.int(nrow(G), n), , drop = FALSE]
        y <- as.numeric(sim_multi_traits(G = X, B = B_mat, h2g = ${h2g}, is_h2g_total = FALSE)$P[, 1])
        list(X = X, z = zfrom(X, y))
      }
      cu <- draw(U_full, dats$n_UKB); ca <- draw(A_full, dats$n_ADSP)
      X_U <- cu$X; X_A <- ca$X; z_U <- cu$z; z_A <- ca$z
      source_of_cohorts <- "re-simulated from the 12 panels"
    }
    n_U <- nrow(X_U); n_A <- nrow(X_A); N <- n_U + n_A
    stopifnot(n_U > 1, n_A > 1)

    ## variants polymorphic in both cohorts
    keep <- which(apply(X_U, 2, sd) > 0 & apply(X_A, 2, sd) > 0 & is.finite(z_U) & is.finite(z_A))
    X_U <- X_U[, keep, drop = FALSE]; X_A <- X_A[, keep, drop = FALSE]
    z_U <- z_U[keep]; z_A <- z_A[keep]
    true_vars <- match(true_vars, keep); true_vars <- true_vars[!is.na(true_vars)]
    z_meta <- (sqrt(n_A) * z_A + sqrt(n_U) * z_U) / sqrt(N)                     # sample-size weighted meta z

    ## ---- (a) in-sample meta-analysis LD (Eq. S54) -----------------------------------------
    R_meta <- (n_A * cor(X_A) + n_U * cor(X_U)) / N

    ## ---- (b) merged raw Gaussian study sketches (Eq. 14 / S55) ----------------------------
    ## U_k = X_k' G_k / sqrt(n_k - 1), X_k standardized, G_k iid N(0, 1)  ->  E[U_k U_k'] / B = R_k
    raw_sketch <- function(X, B) crossprod(scale(X), matrix(rnorm(nrow(X) * B), nrow(X), B)) / sqrt(nrow(X) - 1)

    safe_fit <- function(...) tryCatch(susie_hc(...), error = function(e) { cat("[FAIL]", conditionMessage(e), "\n"); NULL })
    fit_in <- safe_fit(z = z_meta, R = R_meta, n = N)

    per_B <- map(c(${", ".join(str(x) for x in B)}), function(Bk) {
      U_meta <- cbind(sqrt(n_A / Bk) * raw_sketch(X_A, Bk), sqrt(n_U / Bk) * raw_sketch(X_U, Bk))
      R_hat  <- tcrossprod(U_meta) / N
      fit    <- safe_fit(z = z_meta, X = t(U_meta), n = N)                        # baseline SuSiE-RSS on the merged sketch
      list(B = Bk, fit = fit,
           ld_mse = mean((R_hat - R_meta)[upper.tri(R_meta)]^2))
    })

    pip_in <- if (is.null(fit_in)) rep(NA_real_, length(z_meta)) else fit_in$pip
    summ <- map_dfr(per_B, function(x) {
      pip_m <- if (is.null(x$fit)) rep(NA_real_, length(z_meta)) else x$fit$pip
      tibble(B = x$B, pip_cor = suppressWarnings(cor(pip_in, pip_m)),
             max_abs_diff = max(abs(pip_in - pip_m)), ld_mse = x$ld_mse,
             n_cs_insample = length(fit_in$sets$cs), n_cs_merged = length(x$fit$sets$cs))
    })
    print(summ)

    saveRDS(list(input_path = path, cohorts = source_of_cohorts, n_UKB = n_U, n_ADSP = n_A, N = N,
                 variant_id = ref_order[keep], z_meta = z_meta, true_vars = true_vars,
                 pip_insample = pip_in, cs_insample = fit_in$sets$cs,
                 pip_merged = setNames(map(per_B, ~ if (is.null(.x$fit)) NULL else .x$fit$pip), paste0("B_", c(${", ".join(str(x) for x in B)}))),
                 cs_merged  = setNames(map(per_B, ~ .x$fit$sets$cs), paste0("B_", c(${", ".join(str(x) for x in B)}))),
                 summary = summ),
            ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Merged_study_sketch_PIP"
job_dir="${base_dir}/${job_name}"
sim_root="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_Mix_simulation/result"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_c_CAUSAL
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 48:00
#BSUB -o /WORK_DIR/log/JOB_NAME_c_CAUSAL.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_c_CAUSAL.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/30.FigS10.Merged_Study_Sketch_PIP.ipynb" merged_sketch_pip \
    --sim_files $(ls SIM_ROOT/h2g_H2G/causal_CAUSAL/prop_PROP/sample_*_mix.rds | head -N_REP) \
    --h2g H2G --B 500 1000 2000 5000 \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/prop_PROP -j 4
EOF

chmod +x ${job_name}.sh

# both studies present (ADSP share 50%), mid signal, 50 replicates per causal count
h2g=0.04; prop=0.50; n_rep=50
for causal in 1 2 3; do
  output_script="${job_name}_causal_${causal}.sh"
  sed -e "s|CAUSAL|${causal}|g" -e "s|WORK_DIR|${job_dir}|g" -e "s|SIM_ROOT|${sim_root}|g" \
      -e "s|H2G|${h2g}|g" -e "s|PROP|${prop}|g" -e "s|N_REP|${n_rep}|g" -e "s|JOB_NAME|${job_name}|g" \
    "${job_name}.sh" > "${output_script}"
  sb "${output_script}"
done